In [12]:
import pandas as pd
import json
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)

In [13]:
df_terms_final = pd.read_parquet('cleaned_aws_terms.parquet')
df_products = pd.read_parquet('cleaned_aws_products.parquet')

In [14]:
print("Terms shape:", df_terms_final.shape)
print("Products shape:", df_products.shape)
df_products.head(3)

Terms shape: (1898, 5)
Products shape: (2024, 22)


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.endpointType,attributes.usagetype,attributes.operation,attributes.regionCode,attributes.servicename,attributes.vpnType,attributes.group,attributes.groupDescription,attributes.attachmentType,attributes.trafficDirection,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.fromRegionCode,attributes.toRegionCode
0,U3KHECER6QCVQZ6T,Cloud Connectivity,AmazonVPC,Europe (Spain),AWS Region,IPsec,EUS2-VPN-large-Usage-Hours:ipsec.1,CreateVpnConnection,eu-south-2,Amazon Virtual Private Cloud,VPN Large (5 Gbps),None,None,None,None,None,None,None,None,None,None,None
1,3GWW3MJ3JVTNDT23,None,AmazonVPC,Israel (Tel Aviv),AWS Region,None,ILC1-TransitGateway-Hours,TransitGatewayPeering,il-central-1,Amazon Virtual Private Cloud,None,AWSTransitGateway,Hourly charge for Transit Gateway Peering Atta...,Transit Gateway,None,None,None,None,None,None,None,None
2,QNCR32XES4QEB4UB,VPC Peering,AmazonVPC,US West (Oregon),AWS Region,None,USW2-OdbPeering-AZ-In-Bytes,,us-west-2,Amazon Virtual Private Cloud,None,None,None,None,AZ-In-Bytes,None,None,None,None,None,None,None


In [15]:
if 'attributes.servicename' in df_products.columns:
    df_products.drop(columns=['attributes.servicename'], errors='ignore', inplace=True)

print (f"Dimensions (rows, cols): {df_products.shape}")

Dimensions (rows, cols): (2024, 21)


In [16]:
id_columns_vpc = ['sku', 'productFamily', 'attributes.servicecode', 
                'attributes.location', 'attributes.locationType', 'attributes.regionCode']

In [17]:
feature_columns_vpc = ['attributes.usagetype','attributes.operation','attributes.endpointType','attributes.vpnType', 'attributes.attachmentType',
                    'attributes.trafficDirection', 'attributes.transferType', 'attributes.fromLocation', 'attributes.fromLocationType',
                    'attributes.toLocation', 'attributes.toLocationType', 'attributes.fromRegionCode', 'attributes.toRegionCode',
                    'attributes.group', 'attributes.groupDescription']

In [18]:
total_vpc_cols = id_columns_vpc + feature_columns_vpc

remaining_cols_vpc = [col for col in df_products.columns if col not in total_vpc_cols]

df_final_vpc = df_products[total_vpc_cols].copy()
if remaining_cols_vpc:
    df_final_vpc['additional_attributes'] = df_products[remaining_cols_vpc].apply(
        lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
        axis=1
    )
else:
    df_final_vpc['additional_attributes'] = "{}"

In [19]:
df_master_vpc = pd.merge(
    df_final_vpc, 
    df_terms_final, 
    on='sku', 
    how='inner'
).reset_index(drop=True)

In [20]:
df_master_vpc.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.regionCode,attributes.usagetype,attributes.operation,attributes.endpointType,attributes.vpnType,attributes.attachmentType,attributes.trafficDirection,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.fromRegionCode,attributes.toRegionCode,attributes.group,attributes.groupDescription,additional_attributes,rateCode,description,unit,priceUSD
0,U3KHECER6QCVQZ6T,Cloud Connectivity,AmazonVPC,Europe (Spain),AWS Region,eu-south-2,EUS2-VPN-large-Usage-Hours:ipsec.1,CreateVpnConnection,IPsec,VPN Large (5 Gbps),None,None,None,None,None,None,None,None,None,None,None,{},U3KHECER6QCVQZ6T.JRTCKXETXF.6YS6EN2CT7,$0.60 per hour for CreateVpnConnection in Euro...,Hrs,0.600
1,3GWW3MJ3JVTNDT23,None,AmazonVPC,Israel (Tel Aviv),AWS Region,il-central-1,ILC1-TransitGateway-Hours,TransitGatewayPeering,None,None,Transit Gateway,None,None,None,None,None,None,None,None,AWSTransitGateway,Hourly charge for Transit Gateway Peering Atta...,{},3GWW3MJ3JVTNDT23.JRTCKXETXF.6YS6EN2CT7,USD0.055 per hour for TransitGateway-Hours:Tra...,hour,0.055
2,H24SU884XW7MB5WC,None,AmazonVPC,EU (Stockholm),AWS Region,eu-north-1,EUN1-PublicIPv4:InUseAddress,,None,None,None,None,None,None,None,None,None,None,None,VPCPublicIPv4Address,Hourly charge for In-use Public IPv4 Addresses,{},H24SU884XW7MB5WC.JRTCKXETXF.6YS6EN2CT7,$0.005 per In-use public IPv4 address per hour,Hrs,0.005
3,EHGME54GNN6GTD8Y,VpcEndpoint,AmazonVPC,US West (Oregon),AWS Region,us-west-2,USW2-VpcResource-ODB-Consumer-Bytes,VpcResourceConsumer,Resource,None,None,None,None,None,None,None,None,None,None,VpcResources,Per GB charge to access ODB Network Resources,{},EHGME54GNN6GTD8Y.JRTCKXETXF.D7RZ6WCPVW,$0.01 from 0 to 1 PB - Per GB charge to access...,GB,0.010
4,CWACH8635EGQMD9G,None,AmazonVPC,Canada West (Calgary),AWS Region,ca-west-1,CAN2-PublicIPv4:ContiguousBlock,ProvisionIpamPoolCidr,None,None,None,None,None,None,None,None,None,None,None,VPCIPv4ContiguousBlocks,Hourly charge for IPv4 address in a contiguous...,{},CWACH8635EGQMD9G.JRTCKXETXF.6YS6EN2CT7,$0.008 per hour per IPv4 address in contiguous...,Hrs,0.008


#### Bοηθητικός κώδικας

In [21]:
summary_data = []
for col in df_products.columns:
    sample_vals = list(df_products[col].dropna().unique()[:5])
    summary_data.append({'Column': col, 'Sample_Values': sample_vals})

df_summary = pd.DataFrame(summary_data)

# Εμφάνιση χωρίς περικοπές
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df_summary

,Column,Sample_Values
0,sku,"[U3KHECER6QCVQZ6T, 3GWW3MJ3JVTNDT23, QNCR32XES4QEB4UB, H24SU884XW7MB5WC, EHGME54GNN6GTD8Y]"
1,productFamily,"[Cloud Connectivity, VPC Peering, VpcEndpoint, VPC Encryption Controls, VPC Route Server]"
2,attributes.servicecode,[AmazonVPC]
3,attributes.location,"[Europe (Spain), Israel (Tel Aviv), US West (Oregon), EU (Stockholm), US East (Ohio)]"
4,attributes.locationType,"[AWS Region, AWS Local Zone]"
5,attributes.endpointType,"[IPsec, Resource, PrivateLink, Gateway Load Balancer Endpoint, Endpoint Service]"
6,attributes.usagetype,"[EUS2-VPN-large-Usage-Hours:ipsec.1, ILC1-TransitGateway-Hours, USW2-OdbPeering-AZ-In-Bytes, EUN1-PublicIPv4:InUseAddress, USW2-VpcResource-ODB-Consumer-Bytes]"
7,attributes.operation,"[CreateVpnConnection, TransitGatewayPeering, , VpcResourceConsumer, VpcLattice]"
8,attributes.regionCode,"[eu-south-2, il-central-1, us-west-2, eu-north-1, us-east-2]"
9,attributes.vpnType,"[VPN Large (5 Gbps), VPN Concentrator, VPN Concentrator Sites, VPN Standard (1.25 Gbps)]"
